# Narzędzia do interaktywnej eksploracji danych

**Eksploracja danych · laboratorium**

Dzisiaj zbudujemy mały panel analityczny do danych sprzedażowych zapisanych w bazie SQLite. Przejdziemy trzy kroki:

| Część | Temat | Co powstaje |
|---|---|---|
| A | SQLite i pandas | funkcje pobierające i agregujące dane |
| B | DuckDB | te same agregacje w SQL na DataFrame i pliku Parquet |
| C | Plotly Express | interaktywne wykresy i mapa |
| D | ipywidgets | panel z filtrami i formularzem w notebooku |

Na końcu prowadzący pokaże, jak **te same funkcje** zamienić w aplikację (Streamlit, Panel, Mercury).

**Zaliczenie:** uzupełniony notebook w Twoim repozytorium na GitHubie. Komórki oznaczone `# TODO` są do uzupełnienia, a komórki `assert` po nich sprawdzają, czy wynik jest poprawny.

## 0. Przygotowanie środowiska

In [1]:
# W Colabie większość bibliotek jest już zainstalowana; upewniamy się co do wersji
%pip install -q "plotly>=5.24" duckdb

In [2]:
import sqlite3
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

pd.set_option("display.max_columns", 20)

Poniższa komórka tworzy plik `sales.db` z dwiema tabelami: `stores` (sklepy) i `sales` (transakcje).
Uruchom ją, nie musisz jej modyfikować. Można ją uruchamiać wielokrotnie: baza jest za każdym razem tworzona od nowa.

In [3]:
"""Generator bazy sales.db do zajęć z eksploracji danych.

Tworzy dwie tabele:
  stores(store_id, city, latitude, longitude)
  sales(id, product, quantity, price, date, store_id)

Dane są syntetyczne (ziarno losowania stałe), ok. 2500 transakcji
z okresu 1.09-30.11.2026: weekendy są słabsze, jest lekki trend wzrostowy
i Black Friday (27.11). W danych ukryto kilka błędów do znalezienia.
"""
import sqlite3
from datetime import date, timedelta

import numpy as np

STORES = [
    (1, "Łódź", 51.7592, 19.4560),
    (2, "Warszawa", 52.2297, 21.0122),
    (3, "Kraków", 50.0647, 19.9450),
    (4, "Gdańsk", 54.3520, 18.6466),
    (5, "Wrocław", 51.1079, 17.0385),
    (6, "Poznań", 52.4064, 16.9252),
]
STORE_WEIGHTS = [0.15, 0.30, 0.18, 0.10, 0.15, 0.12]

# produkt: (cena bazowa, średnia liczba sztuk w transakcji)
PRODUCTS = {
    "Laptop": (3200, 1.3),
    "Monitor": (850, 2.0),
    "Klawiatura": (150, 3.0),
    "Mysz": (55, 4.0),
    "Słuchawki": (300, 2.0),
    "Kamera": (250, 1.5),
}


def create_sales_db(path="sales.db", n_days=91, seed=42):
    rng = np.random.default_rng(seed)
    conn = sqlite3.connect(path)
    cur = conn.cursor()
    # DROP zamiast CREATE IF NOT EXISTS: ponowne uruchomienie nie dubluje danych
    cur.executescript("""
        DROP TABLE IF EXISTS sales;
        DROP TABLE IF EXISTS stores;
        CREATE TABLE stores (
            store_id  INTEGER PRIMARY KEY,
            city      TEXT NOT NULL,
            latitude  REAL NOT NULL,
            longitude REAL NOT NULL
        );
        CREATE TABLE sales (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            product  TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            price    REAL NOT NULL,
            date     TEXT NOT NULL,
            store_id INTEGER NOT NULL REFERENCES stores(store_id)
        );
    """)
    cur.executemany("INSERT INTO stores VALUES (?, ?, ?, ?)", STORES)

    rows = []
    start = date(2026, 9, 1)
    names = list(PRODUCTS)
    for d in range(n_days):
        day = start + timedelta(days=d)
        factor = 1 + 0.4 * d / n_days               # trend
        if day.weekday() >= 5:                      # weekend
            factor *= 0.6
        if day == date(2026, 11, 27):               # Black Friday
            factor *= 3.0
        for _ in range(rng.poisson(26 * factor)):
            product = rng.choice(names)
            base_price, mean_qty = PRODUCTS[product]
            qty = int(rng.poisson(mean_qty - 1) + 1)
            price = round(base_price * rng.uniform(0.9, 1.1), 2)
            if day == date(2026, 11, 27):
                price = round(price * 0.8, 2)       # promocja
            store = int(rng.choice([s[0] for s in STORES], p=STORE_WEIGHTS))
            rows.append((product, qty, price, day.isoformat(), store))

    # ukryte błędy danych do znalezienia podczas eksploracji
    rows[150] = ("Laptop", 1, 32000.0, rows[150][3], rows[150][4])   # literówka w cenie
    rows[900] = ("Mysz", -5, 52.0, rows[900][3], rows[900][4])       # ujemna ilość
    rows[1700] = ("Monitor", 400, 860.0, rows[1700][3], rows[1700][4])  # podejrzanie duże zamówienie

    cur.executemany(
        "INSERT INTO sales (product, quantity, price, date, store_id) VALUES (?, ?, ?, ?, ?)",
        rows,
    )
    conn.commit()
    conn.close()
    return len(rows)

n = create_sales_db("sales.db")
print(f"Utworzono sales.db: {n} transakcji")

Utworzono sales.db: 2451 transakcji


## A. SQLite i pandas

### A.0 Rozpoznanie bazy
Zanim cokolwiek policzymy, sprawdzamy, **co jest w bazie**: jakie tabele, jakie kolumny, ile rekordów.

In [4]:
conn = sqlite3.connect("sales.db")

# lista tabel zapisana w systemowej tabeli sqlite_master
pd.read_sql("SELECT name, sql FROM sqlite_master WHERE type = 'table'", conn)

,name,sql
0,stores,CREATE TABLE stores (\n store_id I...
1,sales,CREATE TABLE sales (\n id INT...
2,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"


In [5]:
# klasyczne API: kursor + fetchall() zwraca listę krotek
cur = conn.cursor()
cur.execute("SELECT * FROM sales LIMIT 5")
for row in cur.fetchall():
    print(row)

(1, 'Klawiatura', 2, 164.27, '2026-09-01', 5)
(2, 'Kamera', 2, 247.52, '2026-09-01', 2)
(3, 'Monitor', 3, 803.63, '2026-09-01', 3)
(4, 'Kamera', 1, 266.38, '2026-09-01', 4)
(5, 'Laptop', 1, 3501.25, '2026-09-01', 6)


In [6]:
# wygodniej: od razu DataFrame
pd.read_sql("SELECT * FROM stores", conn)

,store_id,city,latitude,longitude
0,1,Łódź,51.7592,19.4560
1,2,Warszawa,52.2297,21.0122
2,3,Kraków,50.0647,19.9450
3,4,Gdańsk,54.3520,18.6466
4,5,Wrocław,51.1079,17.0385
5,6,Poznań,52.4064,16.9252


### A.1 Agregacja: SQL czy pandas?
To samo pytanie, *„ile zarobił każdy produkt?”*, można zadać bazie albo pandasowi. Wynik jest ten sam.

In [7]:
sql_result = pd.read_sql("""
    SELECT product, SUM(quantity * price) AS revenue
    FROM sales
    GROUP BY product
    ORDER BY revenue DESC
""", conn)

raw = pd.read_sql("SELECT * FROM sales", conn)
pandas_result = (raw.assign(revenue=raw.quantity * raw.price)
                    .groupby("product", as_index=False)["revenue"].sum()
                    .sort_values("revenue", ascending=False))

display(sql_result, pandas_result)

,product,revenue
0,Laptop,1866941.32
1,Monitor,1041825.66
2,Słuchawki,244746.23
3,Klawiatura,170774.34
4,Kamera,155176.99
5,Mysz,89181.60


,product,revenue
2,Laptop,1866941.32
3,Monitor,1041825.66
5,Słuchawki,244746.23
1,Klawiatura,170774.34
0,Kamera,155176.99
4,Mysz,89181.60


> **Kiedy co?** Przy dużych danych lepiej, żeby filtrowanie i agregację wykonała baza, bo do Pythona trafia wtedy tylko wynik. Przy małych danych i eksploracji pandas jest wygodniejszy.

### A.2 Zadanie: JOIN
Tabela `sales` ma tylko `store_id`. Miasto i współrzędne są w tabeli `stores`.
Napisz zapytanie, które zwróci **przychód i liczbę transakcji w każdym mieście** (kolumny: `city`, `revenue`, `transactions`), posortowane malejąco po przychodzie.

In [8]:
city_query = """
    SELECT
        stores.city AS city,
        ROUND(SUM(sales.price * sales.quantity), 2) AS revenue,
        COUNT(sales.id) AS transactions
    FROM sales
    LEFT JOIN stores ON stores.store_id = sales.store_id
    GROUP BY stores.store_id, stores.city
    ORDER BY revenue DESC
"""
by_city = pd.read_sql_query(city_query, conn)
by_city

,city,revenue,transactions
0,Warszawa,1327437.04,758
1,Kraków,630091.57,435
2,Łódź,471442.30,353
3,Wrocław,430141.56,348
4,Poznań,388078.81,297
5,Gdańsk,321454.86,260


In [9]:
assert list(by_city.columns) == ["city", "revenue", "transactions"]
assert by_city.iloc[0]["city"] == "Warszawa"
assert by_city["transactions"].sum() == pd.read_sql("SELECT COUNT(*) AS n FROM sales", conn)["n"][0]
print("OK")

OK


### A.3 Zapytania z parametrami
Wartości od użytkownika (np. z widżetu) **nigdy** nie wklejamy do SQL f-stringiem, bo grozi to *SQL injection* i błędami przy apostrofach.
Zamiast tego w zapytaniu stawiamy `?`, a wartości przekazujemy osobno w `params`.

In [10]:
product = "Laptop"

# ŹLE:   f"SELECT * FROM sales WHERE product = '{product}'"
# DOBRZE:
pd.read_sql("SELECT * FROM sales WHERE product = ? AND date >= ?", conn,
            params=[product, "2026-11-01"]).head()

,id,product,quantity,price,date,store_id
0,1482,Laptop,1,3512.74,2026-11-01,2
1,1483,Laptop,1,3119.27,2026-11-01,3
2,1493,Laptop,2,2954.70,2026-11-02,3
3,1495,Laptop,1,3293.55,2026-11-02,6
4,1501,Laptop,1,2910.39,2026-11-02,1


### A.4 Zadanie: funkcje warstwy logiki
Te trzy funkcje będą używane do końca zajęć: przez wykresy, widżety, a na koniec przez aplikacje.

1. `get_sales(conn, products=None, date_from=None, date_to=None)` zwraca DataFrame z kolumnami
   `id, product, quantity, price, revenue, date, city, latitude, longitude`, gdzie `revenue = quantity * price`.
   Filtry są opcjonalne, gdy podano `None`, nie filtrujemy. Użyj parametrów `?`. Kolumnę `date` zamień na `datetime`.
2. `daily_revenue(df)` zwraca kolumny `date, revenue` (suma przychodu w każdym dniu).
3. `revenue_by_product(df)` zwraca kolumny `product, quantity, revenue` posortowane malejąco po `revenue`.

*Wskazówka do filtra listy produktów:* `"IN (" + ", ".join("?" * len(products)) + ")"` daje `IN (?, ?, ?)`.

In [11]:
def get_sales(conn, products=None, date_from=None, date_to=None):
    clauses = []
    query_params = []

    if products is not None:
        selected = tuple(products)
        if len(selected) == 0:
            clauses.append("0 = 1")
        else:
            marks = ", ".join("?" for _ in selected)
            clauses.append(f"sales.product IN ({marks})")
            query_params.extend(selected)

    if date_from is not None:
        clauses.append("sales.date >= ?")
        query_params.append(pd.Timestamp(date_from).strftime("%Y-%m-%d"))

    if date_to is not None:
        clauses.append("sales.date <= ?")
        query_params.append(pd.Timestamp(date_to).strftime("%Y-%m-%d"))

    where_sql = f"WHERE {' AND '.join(clauses)}" if clauses else ""
    sql = f"""
        SELECT
            sales.id,
            sales.product,
            sales.quantity,
            sales.price,
            sales.quantity * sales.price AS revenue,
            sales.date,
            stores.city,
            stores.latitude,
            stores.longitude
        FROM sales
        JOIN stores ON sales.store_id = stores.store_id
        {where_sql}
        ORDER BY sales.date, sales.id
    """

    frame = pd.read_sql_query(sql, conn, params=query_params)
    frame["date"] = pd.to_datetime(frame["date"])
    return frame


def daily_revenue(df):
    return (
        df.groupby("date")["revenue"]
        .sum()
        .reset_index()
        .sort_values(by="date")
    )


def revenue_by_product(df):
    return (
        df.groupby("product")[["quantity", "revenue"]]
        .sum()
        .reset_index()
        .sort_values(by="revenue", ascending=False, ignore_index=True)
    )

In [12]:
df = get_sales(conn)
assert len(df) == pd.read_sql("SELECT COUNT(*) AS n FROM sales", conn)["n"][0]
assert pd.api.types.is_datetime64_any_dtype(df["date"])

part = get_sales(conn, products=["Mysz", "Kamera"], date_from="2026-10-01", date_to="2026-10-31")
assert set(part["product"]) == {"Mysz", "Kamera"}
assert part["date"].min() >= pd.Timestamp("2026-10-01") and part["date"].max() <= pd.Timestamp("2026-10-31")

assert list(daily_revenue(df).columns) == ["date", "revenue"]
assert list(revenue_by_product(df).columns) == ["product", "quantity", "revenue"]
assert revenue_by_product(df).iloc[0]["product"] == "Laptop"
print("OK")

OK


### A.5 Zadanie: czy dane są czyste?
W bazie ukryto **trzy** błędne lub podejrzane rekordy. Znajdź je (np. `describe()`, sortowanie, warunki logiczne) i krótko opisz w komórce Markdown, co jest z nimi nie tak.
Na razie ich **nie usuwaj**: zobaczysz, jak wyglądają na wykresach.

In [13]:
print("Statystyki opisowe kolumn liczbowych:")
display(df[["quantity", "price", "revenue"]].describe().T)

print("\nWyłapane rekordy odstające:")
bad_rows = df.query("quantity <= 0 or quantity > 50 or price > 10000")
display(bad_rows[["id", "date", "city", "product", "quantity", "price", "revenue"]])

Statystyki opisowe kolumn liczbowych:


,count,mean,std,min,25%,50%,75%,max
quantity,2451.0,2.466748,8.160883,-5.00,1.00,2.00,3.000,400.0
price,2451.0,842.106614,1286.018170,40.59,149.97,273.19,861.675,32000.0
revenue,2451.0,1455.995977,7147.913321,-260.00,273.26,570.36,1727.470,344000.0



Wyłapane rekordy odstające:


,id,date,city,product,quantity,price,revenue
150,151,2026-09-08,Warszawa,Laptop,1,32000.0,32000.0
900,901,2026-10-10,Warszawa,Mysz,-5,52.0,-260.0
1700,1701,2026-11-09,Warszawa,Monitor,400,860.0,344000.0


*Twoje wnioski:*
- **id = 151 (2026-09-08, Laptop):** cena jednostkowa wynosi `32 000 zł` przy normalnych cenach ok. `3 200 zł` — typowa literówka (jedno zero za dużo).
- **id = 901 (2026-10-10, Mysz):** ujemna liczba sztuk (`-5`), co daje ujemny przychód `-260 zł` — błąd wprowadzania albo nieprawidłowo zapisany zwrot.
- **id = 1701 (2026-11-09, Monitor):** aż `400` sztuk w pojedynczej transakcji (`344 000 zł`), mimo że pozostałe zamówienia mieszczą się w kilku sztukach — albo gruby błąd w ilości, albo nietypowa sprzedaż B2B zaburzająca średnie.

## B. DuckDB: SQL bez serwera, prosto na danych

[DuckDB](https://duckdb.org) to analityczna (kolumnowa) baza danych działająca, podobnie jak SQLite, w procesie Pythona.
Różnica: SQLite jest zoptymalizowany pod wiele małych zapisów (aplikacje), a DuckDB pod szybkie agregacje dużych zbiorów (analiza).
DuckDB umie wykonać SQL **bezpośrednio na DataFrame**, na plikach CSV i Parquet, bez wczytywania ich do bazy.

In [14]:
import duckdb

# SQL na DataFrame `df`: DuckDB znajduje zmienną po nazwie
duckdb.sql("""
    SELECT city, product, ROUND(SUM(revenue)) AS revenue
    FROM df
    GROUP BY city, product
    ORDER BY revenue DESC
    LIMIT 5
""").df()

,city,product,revenue
0,Warszawa,Laptop,569996.0
1,Warszawa,Monitor,558328.0
2,Kraków,Laptop,376466.0
3,Łódź,Laptop,290558.0
4,Wrocław,Laptop,234507.0


In [15]:
# zapis do Parquet i zapytanie bezpośrednio na pliku
df.to_parquet("sales.parquet", index=False)

duckdb.sql("""
    SELECT date_trunc('week', date) AS week, SUM(revenue) AS revenue
    FROM 'sales.parquet'
    GROUP BY week
    ORDER BY week
""").df().head()

,week,revenue
0,2026-08-31,166269.22
1,2026-09-07,244789.09
2,2026-09-14,239404.00
3,2026-09-21,188996.48
4,2026-09-28,223989.76


### B.1 Zadanie
Używając DuckDB na pliku `sales.parquet`, policz dla każdego **dnia tygodnia** średni dzienny przychód.
Czy weekendy różnią się od dni roboczych? Który dzień wypada podejrzanie wysoko i dlaczego (wróć do A.5)?

*Wskazówki:* `dayname(date)` lub `isodow(date)`; najpierw zsumuj przychód w dniu (podzapytanie / `WITH`), potem uśrednij.

In [16]:
dow_summary = duckdb.sql("""
    WITH per_day AS (
        SELECT
            CAST(date AS DATE) AS sale_day,
            SUM(revenue) AS day_total
        FROM 'sales.parquet'
        GROUP BY sale_day
    )
    SELECT
        isodow(sale_day) AS nr_dnia,
        dayname(sale_day) AS dzien_tygodnia,
        ROUND(AVG(day_total), 2) AS sredni_przychod,
        ROUND(MAX(day_total), 2) AS maks_dzienny
    FROM per_day
    GROUP BY nr_dnia, dzien_tygodnia
    ORDER BY nr_dnia
""").df()


dow_summary

,nr_dnia,dzien_tygodnia,sredni_przychod,maks_dzienny
0,1,Monday,65880.50,407602.46
1,2,Tuesday,43716.63,60787.20
2,3,Wednesday,34679.08,74253.02
3,4,Thursday,41314.62,55169.08
4,5,Friday,42216.05,86755.08
5,6,Saturday,24989.38,39410.58
6,7,Sunday,21714.99,31243.53


## C. Plotly Express: interaktywne wykresy

Pierwszy poziom interaktywności nie wymaga żadnego kodu: wykres Plotly ma tooltip po najechaniu, zoom, przesuwanie i klikalną legendę.
Składnia `px.<typ>(df, x=..., y=..., color=...)` przypomina seaborn.

In [17]:
fig = px.line(daily_revenue(df), x="date", y="revenue", markers=True,
              title="Sprzedaż dzienna [zł]",
              labels={"date": "Data", "revenue": "Przychód [zł]"})
fig.show()

Najedź na największy pik i powiększ go myszą. Który rekord z zadania A.5 go powoduje? \
Największy skok przypada na **2026-11-09** i wynika z transakcji na **400 monitorów** (`id = 1701`, wartość `344 000 zł`).

### C.1 Zadanie: wykres słupkowy
Narysuj liczbę **sprzedanych sztuk** wg produktu (`revenue_by_product`). Dodaj przychód do tooltipa (`hover_data`) i polskie podpisy osi (`labels`).

In [18]:
by_prod = revenue_by_product(df)

fig_qty = px.bar(
    by_prod,
    x="product",
    y="quantity",
    text="quantity",
    hover_data={"revenue": ":,.2f"},
    title="Sprzedane sztuki w podziale na produkty",
    labels={
        "product": "Produkt",
        "quantity": "Sprzedane sztuki",
        "revenue": "Łączny przychód [zł]",
    },
)
fig_qty.update_traces(textposition="outside")
fig_qty.show()

### C.2 Zadanie: mapa sklepów
Zagreguj dane po `city, latitude, longitude` (suma `revenue`, liczba transakcji) i pokaż sklepy na mapie za pomocą `px.scatter_map`.
Wielkość punktu ma odpowiadać przychodowi.

In [19]:
def aggregate_stores(frame):
    return (
        frame.groupby(["city", "latitude", "longitude"], as_index=False)
        .agg(revenue=("revenue", "sum"), transactions=("id", "count"))
    )


store_summary = aggregate_stores(df)
fig_map = px.scatter_map(
    store_summary,
    lat="latitude",
    lon="longitude",
    size="revenue",
    size_max=38,
    hover_name="city",
    hover_data={
        "revenue": ":,.2f",
        "transactions": True,
        "latitude": False,
        "longitude": False,
    },
    zoom=5,
    center={"lat": 52.1, "lon": 19.1},
    title="Przychód i liczba transakcji w sklepach",
    labels={"revenue": "Przychód [zł]", "transactions": "Liczba transakcji"},
)
fig_map.show()

### C.3 Zadanie (krótkie): jeden wykres, więcej informacji
Narysuj sprzedaż dzienną **osobno dla każdego produktu** (jedna linia na produkt, `color="product"`).
Kliknij dwukrotnie produkt w legendzie: co się dzieje?

In [20]:
per_product_daily = (
    df.groupby(["date", "product"])["revenue"]
    .sum()
    .reset_index()
)

fig_daily_prod = px.line(
    per_product_daily,
    x="date",
    y="revenue",
    color="product",
    title="Dzienna sprzedaż z podziałem na produkty",
    labels={"date": "Data", "revenue": "Przychód [zł]", "product": "Produkt"},
)
fig_daily_prod.show()
# Podwójne kliknięcie pozycji w legendzie izoluje tylko ten jeden produkt (ukrywa resztę).

## D. ipywidgets: panel w notebooku

Drugi poziom: **własne kontrolki**. Biblioteka `ipywidgets` daje suwaki, listy, przyciski i pola tekstowe, które działają w Colabie i Jupyterze.

### D.0 Najszybciej: `interact`
`interact` sam dobiera widżet do typu argumentu: lista daje rozwijaną listę, a krotka liczb suwak.

In [21]:
@widgets.interact(product=get_sales(conn)["product"].unique().tolist(), min_qty=(1, 10))
def show(product, min_qty):
    sub = df[(df["product"] == product) & (df["quantity"] >= min_qty)]
    print(f"{len(sub)} transakcji, przychód {sub['revenue'].sum():,.0f} zł")
    px.histogram(sub, x="price", nbins=30, title=f"Rozkład cen: {product}").show()

interactive(children=(Dropdown(description='product', options=('Klawiatura', 'Kamera', 'Monitor', 'Laptop', 'S…

### D.1 Budowa panelu krok po kroku
Pełny panel składa się z trzech elementów:
1. **widżety**, które przechowują stan (`.value`),
2. **funkcja `update`**, która czyta `.value`, wywołuje funkcje z części A i C i rysuje wynik w `Output`,
3. **połączenie**: `widget.observe(update, names="value")` wywołuje `update` przy każdej zmianie.

In [22]:
products = sorted(df["product"].unique())

w_products = widgets.SelectMultiple(options=products, value=products, description="Produkty", rows=6)
w_from = widgets.DatePicker(description="Od", value=pd.Timestamp("2026-09-01").date())
w_to = widgets.DatePicker(description="Do", value=pd.Timestamp("2026-11-30").date())
out = widgets.Output()

widgets.VBox([w_products, w_from, w_to])   # podgląd samych kontrolek

### D.2 Zadanie: funkcja `update`
Uzupełnij `update`, tak aby po każdej zmianie filtra:
- pobrała dane przez `get_sales(...)` z wartościami widżetów,
- wypisała liczbę transakcji i łączny przychód,
- narysowała wykres dzienny, słupkowy i mapę (możesz skopiować kod z części C).

Następnie podłącz `update` do wszystkich trzech widżetów.

In [23]:
def update(change=None):
    with out:
        out.clear_output(wait=True)

        chosen_products = list(w_products.value)
        if not chosen_products:
            print("Zaznacz chociaż jeden produkt, żeby wyświetlić dane.")
            return

        if w_from.value and w_to.value and w_from.value > w_to.value:
            print("Niepoprawny przedział dat: data początkowa musi być wcześniejsza niż końcowa.")
            return

        subset = get_sales(
            conn,
            products=chosen_products,
            date_from=w_from.value,
            date_to=w_to.value,
        )

        if subset.empty:
            print("Brak wyników dla podanych filtrów.")
            return

        n_tx = len(subset)
        rev_sum = subset["revenue"].sum()
        print(f"Transakcje: {n_tx} | Suma przychodu: {rev_sum:,.2f} zł")

        px.line(
            daily_revenue(subset),
            x="date",
            y="revenue",
            markers=True,
            title="Sprzedaż dzienna [zł]",
            labels={"date": "Data", "revenue": "Przychód [zł]"},
        ).show()

        px.bar(
            revenue_by_product(subset),
            x="product",
            y="quantity",
            text="quantity",
            hover_data={"revenue": ":,.2f"},
            title="Sprzedane sztuki wg produktu",
            labels={"product": "Produkt", "quantity": "Sztuki", "revenue": "Przychód [zł]"},
        ).show()

        px.scatter_map(
            aggregate_stores(subset),
            lat="latitude",
            lon="longitude",
            size="revenue",
            size_max=38,
            hover_name="city",
            hover_data={"revenue": ":,.2f", "transactions": True, "latitude": False, "longitude": False},
            zoom=5,
            center={"lat": 52.1, "lon": 19.1},
            title="Przychód wg sklepu",
            labels={"revenue": "Przychód [zł]", "transactions": "Liczba transakcji"},
        ).show()


for widget in [w_products, w_from, w_to]:
    widget.observe(update, names="value")

update()
panel = widgets.HBox([widgets.VBox([w_products, w_from, w_to]), out])
display(panel)

### D.3 Zadanie: formularz dodawania sprzedaży
Panel eksploracyjny często pozwala też **poprawić lub dopisać dane**. Zbuduj formularz:
`Dropdown` (produkt), `IntText` (ilość), `FloatText` (cena), `DatePicker` (data), `Dropdown` (sklep: etykieta miasta, wartość `store_id`) i `Button` „Dodaj”.

Po kliknięciu:
- sprawdź, czy ilość i cena są dodatnie (jeśli nie, wypisz komunikat i nic nie zapisuj),
- wstaw rekord zapytaniem `INSERT` z parametrami `?` i zrób `conn.commit()`,
- wywołaj `update()`, żeby panel z D.2 się odświeżył.

*Wskazówka:* `Dropdown(options=[("Łódź", 1), ("Warszawa", 2), ...])` pokazuje nazwę, a w `.value` jest liczba.

In [24]:
store_rows = pd.read_sql_query("SELECT store_id, city FROM stores ORDER BY store_id", conn)
city_choices = [(row.city, int(row.store_id)) for row in store_rows.itertuples()]

inp_product = widgets.Dropdown(options=products, description="Produkt")
inp_qty = widgets.IntText(value=1, description="Ilość")
inp_price = widgets.FloatText(value=50.0, description="Cena [zł]")
inp_date = widgets.DatePicker(description="Data", value=pd.Timestamp("2026-11-30").date())
inp_store = widgets.Dropdown(options=city_choices, description="Sklep")
submit_btn = widgets.Button(description="Dodaj", button_style="info", icon="check")
feedback_out = widgets.Output()


def handle_submit(_):
    with feedback_out:
        feedback_out.clear_output()

        if inp_qty.value <= 0:
            print("Błąd: ilość musi być większa od 0.")
            return
        if inp_price.value <= 0:
            print("Błąd: cena musi być większa od 0.")
            return
        if inp_date.value is None:
            print("Błąd: podaj datę transakcji.")
            return

        with conn:
            cur = conn.execute(
                "INSERT INTO sales (product, quantity, price, date, store_id) VALUES (?, ?, ?, ?, ?)",
                (
                    inp_product.value,
                    int(inp_qty.value),
                    float(inp_price.value),
                    inp_date.value.isoformat(),
                    int(inp_store.value),
                ),
            )
        print(
            f"Dodano wpis #{cur.lastrowid}: {inp_product.value}, "
            f"{inp_qty.value} szt. x {inp_price.value:.2f} zł ({inp_date.value})."
        )
        update()


submit_btn.on_click(handle_submit)
display(widgets.VBox([inp_product, inp_qty, inp_price, inp_date, inp_store, submit_btn, feedback_out]))

## E. Co dalej: od notebooka do aplikacji

Panel z części D działa tylko dla kogoś, kto uruchomi notebook. Żeby udostępnić go osobie, która nie zna Pythona, potrzebujemy **aplikacji**.
Ponieważ logika (`get_sales`, `daily_revenue`, …) jest oddzielona od widżetów, zmienia się tylko warstwa interfejsu:

| Narzędzie | Wejście | Uruchomienie |
|---|---|---|
| **Streamlit** | skrypt `app.py` | `streamlit run app.py` |
| **Panel** | skrypt lub notebook | `panel serve app.py` |
| **Mercury** | notebook z widżetami `mr.*` | `mercury` (w katalogu z notebookiem) |

Gotowe przykłady pokazane na zajęciach: `app_streamlit.py`, `app_panel.py`, `demo_mercury.ipynb` (wymagają pliku `sales_logic.py` z tymi samymi funkcjami).
To dobry punkt startu, jeśli projekt na wyższą ocenę ma mieć formę interaktywnej aplikacji.

### Na koniec
- [ ] wszystkie komórki `assert` przechodzą,
- [ ] w A.5 opisane są trzy podejrzane rekordy,
- [ ] notebook jest zapisany w Twoim repozytorium na GitHubie.

In [25]:
conn.close()